# Docling exploration — a diagnostic client of `engineering_rag`

**This notebook contains no parsing implementation.** It imports the production package and
calls its public API. A test (`tests/unit/test_architecture.py`) fails the build if this
notebook ever constructs `DocumentConverter`, `PdfPipelineOptions` or `PdfFormatOption`
directly — parsing logic belongs in `src/`, where it is typed, linted and tested.

What it is for:

1. Inspect a PDF **before** conversion (preflight, no Docling involved).
2. See the Docling options the chosen profile actually resolves to.
3. Run the pipeline and look at representative document elements.
4. Read the page-level metrics and understand how to interpret the validation report.

**Kernel:** select *Python (engineering-rag-parser)* — see the README for registration.


## 0. Setup

Paths below resolve from the repository root.


In [ ]:
import json
import os
from pathlib import Path

# Work from the repository root whether the kernel starts here or in notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

PDF = ROOT / "data" / "input" / "Instrumentation-and-Control-Engineering.pdf"
print("repo root :", ROOT)
print("input pdf :", "present" if PDF.is_file() else "MISSING (copy it there first)")

In [ ]:
import engineering_rag as erp
from engineering_rag.services.parser.converter import docling_versions

print("parser version:", erp.__version__)
for k, v in docling_versions().items():
    print(f"  {k:24s} {v}")

## 1. Preflight — the independent baseline

`inspect_source()` measures the PDF with **pypdf + pdfminer.six + pypdfium2**. Docling is not
involved, and that is the whole point: if the baseline were produced by Docling, every
coverage metric later would be Docling agreeing with itself and would prove nothing.


In [ ]:
from engineering_rag.services.parser.config import load_config
from engineering_rag.services.parser.preflight import inspect_source

config = load_config("configs/high_fidelity.yaml")
manifest = inspect_source(PDF, config)

print(f"pages           : {manifest.page_count}")
print(f"sha256          : {manifest.sha256}")
print(f"bytes           : {manifest.byte_size:,}")
print(f"pdf version     : {manifest.pdf_version}   encrypted={manifest.is_encrypted}")
print(f"characters      : {manifest.total_char_count:,}")
print(f"outline entries : {len(manifest.outline_entries)}")
print(f"fonts           : {manifest.fonts}")

### 1.1 Decorative furniture vs substantive figures

The document repeats a branded banner and a watermark on every page. Preflight separates
them from real figures by `(bounding box, pixel size)` **signature repetition**: a signature
recurring across the document is furniture; one appearing once is a figure.

This matters — a naive area threshold would silently discard genuinely small diagrams.


In [ ]:
print(f"raster images total  : {manifest.total_image_count}")
print(f"  decorative repeats : {manifest.decorative_image_count}")
print(f"  substantive figures: {manifest.substantive_image_count}")
print()
print("pages carrying a figure  :", manifest.pages_with_substantive_images)
print("sparse-text pages        :", manifest.sparse_pages)
print("flagged for visual review:", manifest.visual_review_pages)
print()
for c in manifest.furniture_candidates:
    print(f"  {c.kind:12s} {c.band:6s} on {c.page_fraction:.0%} of pages  {c.text[:56]!r}")

### 1.2 Per-page source measurements

Note that `substantive_image_count` — not text volume — is what flags a page for review.


In [ ]:
hdr = f"{'pg':>3} {'chars':>6} {'words':>6} {'lines':>5} {'img':>4} {'figs':>4} {'area':>7}  review"
print(hdr)
print("-" * len(hdr))
for p in manifest.pages:
    flag = "YES" if p.needs_visual_review else ""
    print(
        f"{p.page_no:>3} {p.char_count:>6} {p.word_count:>6} {p.line_count:>5} "
        f"{p.image_count:>4} {p.substantive_image_count:>4} "
        f"{p.substantive_image_area_fraction:>7.1%}  {flag}"
    )

## 2. The Docling options this profile actually resolves to

`describe_effective_options()` **introspects the constructed objects** rather than reciting
documentation, so what you see below is what will actually run.

Two values are worth pausing on:

- `do_ocr = False`. Docling 2.121.0 ships `do_ocr=True`. For a digitally generated,
  text-searchable PDF, OCR duplicates and degrades clean embedded text, so this project
  overrides it explicitly.
- `device = cpu`. Deliberate — see ADR-002 in `TASKS.md`.


In [ ]:
from engineering_rag.services.parser.converter import describe_effective_options
from engineering_rag.services.parser.profiles import choose_profile, resolve_profile_config

decision = choose_profile(manifest, config)
print("profile :", decision.profile.value)
print("reason  :", decision.reason)
print("evidence:", json.dumps(decision.evidence, indent=2))

effective = resolve_profile_config(config, decision.profile)
opts = describe_effective_options(effective)["pipeline_options"]
print()
keys = [
    "do_ocr",
    "do_table_structure",
    "table_structure_options",
    "images_scale",
    "generate_page_images",
    "generate_picture_images",
    "do_picture_description",
    "accelerator_options",
    "document_timeout",
    "enable_remote_services",
]
for key in keys:
    print(f"  {key:26s} = {opts.get(key)}")

## 3. Run the pipeline

One call does preflight -> convert -> export -> validate -> manifest and writes an immutable
run directory. On 4 CPU cores this takes roughly 5-6 minutes for the 27-page document.

Set `RUN_IT = True` to execute; otherwise the most recent existing run is reused.


In [ ]:
from engineering_rag.pipelines.parsing_pipeline import run_parsing_pipeline

RUN_IT = False  # flip to True to actually convert

if RUN_IT:
    result = run_parsing_pipeline(PDF, config, "data/output/parser")
    run_dir = result.run_dir
    print(result.status.value, "->", run_dir)
else:
    runs = sorted(Path("data/output/parser/Instrumentation-and-Control-Engineering").glob("*/"))
    run_dir = runs[-1] if runs else None
    print("using existing run:", run_dir)

## 4. Representative document elements

Load the canonical JSON back through the current `DoclingDocument` model — the same reload
the validation gate performs.


In [ ]:
from docling_core.types.doc import ContentLayer, DocItemLabel, DoclingDocument

doc = DoclingDocument.load_from_json(run_dir / "docling" / "document.json")
print("pages   :", len(doc.pages))
print("texts   :", len(doc.texts))
print("tables  :", len(doc.tables))
print("pictures:", len(doc.pictures))

In [ ]:
# Headings, with their section numbering preserved.
shown = 0
layers = {ContentLayer.BODY}
for item, _lvl in doc.iterate_items(with_groups=False, included_content_layers=layers):
    if getattr(item, "label", None) == DocItemLabel.SECTION_HEADER:
        page = item.prov[0].page_no if item.prov else "?"
        print(f"  L{getattr(item, 'level', '?')} p{page:>3}  {item.text[:64]}")
        shown += 1
        if shown >= 20:
            break

In [ ]:
# Tables. On this document all three table BODIES are raster images, so TableFormer
# recovers zero cells: the region is real, but its content is not text.
for i, t in enumerate(doc.tables):
    page = t.prov[0].page_no if t.prov else "?"
    cells = len(t.data.table_cells or [])
    print(
        f"  table {i}: page {page}  {t.data.num_rows}x{t.data.num_cols}  "
        f"cells={cells}  caption={t.caption_text(doc)!r}"
    )

In [ ]:
# Provenance is exactly what a future RAG citation will point at.
for item, _lvl in doc.iterate_items(with_groups=False, included_content_layers=layers):
    if getattr(item, "text", "") and item.prov:
        p = item.prov[0]
        b = p.bbox
        print(f"  page {p.page_no}  bbox=({b.l:.0f},{b.b:.0f},{b.r:.0f},{b.t:.0f})")
        print(f"  text : {item.text[:70]!r}")
        break

## 5. Exported artifacts


In [ ]:
md_path = run_dir / "markdown" / "document.md"
raw = md_path.read_bytes()
text = raw.decode("utf-8")

print("bytes       :", f"{len(raw):,}")
print("LF only     :", b"\r" not in raw)
print("image links :", text.count("]("))
print("page anchors:", text.count("<!-- page:"))
print("base64      :", text.count("base64,"))
print()
print(text[:900])

In [ ]:
# Referenced assets, and proof that every link resolves.
import re

links = re.findall(r"!\[[^\]]*\]\(([^)]+)\)", text)
for link in links:
    status = "OK    " if (run_dir / link).is_file() else "BROKEN"
    print(f"  {status}  {link}")

## 6. Page-level metrics and how to read them

Several metrics exist because **a single ratio is easy to be misled by**:

| Metric | What it answers | Where it misleads on its own |
|---|---|---|
| `char_coverage` | Did roughly the right volume of text survive? | Legitimately >1.0 or <1.0 when a paragraph crosses a page break |
| `token_recall` | Which *word types* survived? | Ignores whether the important ones did |
| `critical_token_recall` | Did numbers, units and instrument tags survive? | Page-local only, so a moved token looks lost |
| `relocated_*` | Was it moved rather than lost? | — |
| `needs_visual_review` | Is this page's content visual? | — |

A page of pure diagram can score 1.00 on every text metric while conveying nothing. That is
why `needs_visual_review` is presence-driven and entirely independent of the text scores.


In [ ]:
report = json.loads((run_dir / "validation" / "report.json").read_text(encoding="utf-8"))

hdr = f"{'pg':>3} {'src':>5} {'parsed':>6} {'cov':>6} {'tokR':>6} {'critR':>6} {'sev':>8}  notes"
print(hdr)
print("-" * 92)
for r in report["page_coverage"]:
    note = ""
    if r["relocated_spans"] or r["relocated_critical_tokens"]:
        note += "relocated "
    if r["missing_spans"]:
        note += f"MISSING({len(r['missing_spans'])}) "
    print(
        f"{r['page_no']:>3} {r['source_chars']:>5} {r['parsed_chars']:>6} "
        f"{r['char_coverage']:>6.2f} {r['token_recall']:>6.2f} "
        f"{r['critical_token_recall']:>6.2f} {r['severity']:>8}  {note}"
    )

## 7. Interpreting the validation report

- **`PASS`** — every acceptance gate passed, no warnings.
- **`PASS_WITH_WARNINGS`** — every *gate* passed; non-critical uncertainty remains. Read
  `human_review_items`. This is the honest outcome for this document.
- **`FAIL`** — a critical gate failed. Do not use the output; the CLI exits non-zero.

`--strict` escalates warnings to failure for CI.

The distinction that matters most here: `critical_token_recall` is a **page-local WARNING**,
while `document_text_completeness` is a **CRITICAL gate**. Content that moves between pages
is a provenance nuance; content missing from the whole document is a defect.


In [ ]:
print("STATUS:", report["status"])
print()
gates = [c for c in report["checks"] if c["gate"]]
print(f"gates passed: {sum(c['passed'] for c in gates)}/{len(gates)}")
print()
for c in report["checks"]:
    if not c["passed"]:
        tag = c["severity"] + (" GATE" if c["gate"] else "")
        print(f"  [{tag}] {c['check_id']}")
        print(f"      {c['summary'][:150]}")
        print(f"      remediation: {c['remediation'][:120]}")
        print()

In [ ]:
print("HUMAN REVIEW REQUIRED")
for item in report["human_review_items"]:
    print(" *", item)

### 7.1 Visual review artifacts

Each flagged page gets a self-contained HTML card: the source rendering with Docling's
parsed bounding boxes drawn over it, plus that page's metrics.

**A machine cannot confirm that a P&ID's labels and connections were recovered.** These
cards exist to make the human check cheap enough that it actually happens.


In [ ]:
reviews = sorted((run_dir / "validation" / "review").glob("*.html"))
print(f"{len(reviews)} review artifact(s):")
for r in reviews:
    print("  ", r.relative_to(run_dir).as_posix())
print()
if reviews:
    print("Open in a browser, e.g.:")
    print("  file:///" + reviews[0].resolve().as_posix())

## 8. What the next stage consumes

Chunking, embeddings and retrieval are **out of scope** for this milestone. The contract
those stages will consume is already present in the artifacts:

| Field | Where it lives |
|---|---|
| stable document id / source hash | `run_manifest.json` -> `source.sha256` |
| block id, type, heading path | `docling/document.json` (`self_ref`, `label`, element tree) |
| page number + bounding box | every item's `prov[]` |
| table representation | `TableItem.data`, or the preserved asset when unrecovered |
| asset reference | `assets/pictures/*.png` |
| parser + config version | `run_manifest.json` -> `parser_version`, `config_hash` |
| validation status | `validation/report.json` -> `status` |

Structure-aware chunking should consume **`document.json`**, not the Markdown: the JSON
keeps bounding boxes and the element tree that Markdown serialization flattens away.


In [ ]:
m = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("document id (sha256):", m["source"]["sha256"])
print("parser version      :", m["parser_version"])
print("config hash         :", m["config_hash"])
print("profile             :", m["profile"])
print("status              :", m["status"])
print("artifacts hashed    :", len(m["artifacts"]))